In [ ]:
# ============================================================
# Cell 2 — Imports, utilities, and logging helpers
# ============================================================
from __future__ import annotations

import os
import json
import math
import random
from dataclasses import dataclass, asdict, is_dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def json_safe(obj: Any) -> Any:
    """
    Convert numpy / torch / dataclass objects into JSON-safe objects.
    """
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if torch.is_tensor(obj):
        return obj.detach().cpu().tolist()
    if is_dataclass(obj):
        return json_safe(asdict(obj))
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [json_safe(x) for x in obj]
    return str(obj)


class TableLogger:
    def __init__(self):
        self.rows: List[Dict[str, Any]] = []

    def add(self, **kwargs):
        self.rows.append({k: json_safe(v) for k, v in kwargs.items()})

    def save_csv(self, path: str):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        pd.DataFrame(self.rows).to_csv(path, index=False)

    def clear(self):
        self.rows.clear()


def clip(v: np.ndarray, lo: np.ndarray, hi: np.ndarray) -> np.ndarray:
    return np.minimum(np.maximum(v, lo), hi)


def norm(v: np.ndarray, eps: float = 1e-8) -> float:
    return float(np.linalg.norm(v))


def normalize(v: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    n = norm(v, eps)
    if n < eps:
        return np.zeros_like(v, dtype=np.float32)
    return (v / n).astype(np.float32)


def rect_contains(p: np.ndarray, mn: np.ndarray, mx: np.ndarray) -> bool:
    return bool(np.all(p >= mn) and np.all(p <= mx))


def rect_center(mn: np.ndarray, mx: np.ndarray) -> np.ndarray:
    return ((mn + mx) / 2.0).astype(np.float32)


def rect_size(mn: np.ndarray, mx: np.ndarray) -> np.ndarray:
    return (mx - mn).astype(np.float32)

In [ ]:
# ============================================================
# Cell 3 — PPO + LSTM recurrent policy
# ============================================================
class PPORecurrentPolicy(nn.Module):
    def __init__(
        self,
        obs_dim: int,
        action_dim: int,
        hidden_size: int = 256,
        lstm_hidden_size: int = 256,
        lstm_layers: int = 1,
        device: Optional[str] = None,
    ):
        super().__init__()
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.hidden_size = hidden_size
        self.lstm_hidden_size = lstm_hidden_size
        self.lstm_layers = lstm_layers
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )

        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
        )

        self.actor_mean = nn.Linear(lstm_hidden_size, action_dim)
        self.actor_log_std = nn.Parameter(torch.zeros(action_dim))
        self.critic = nn.Linear(lstm_hidden_size, 1)

        self.to(self.device)

    def init_hidden(self, batch_size: int = 1, device: Optional[torch.device] = None):
        device = device or self.device
        h = torch.zeros(self.lstm_layers, batch_size, self.lstm_hidden_size, device=device)
        c = torch.zeros(self.lstm_layers, batch_size, self.lstm_hidden_size, device=device)
        return (h, c)

    def _ensure_2d_obs(self, obs: torch.Tensor) -> torch.Tensor:
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)
        return obs

    def forward_step(
        self,
        obs: torch.Tensor,
        hidden: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    ):
        obs = self._ensure_2d_obs(obs).to(self.device)
        if hidden is None:
            hidden = self.init_hidden(batch_size=obs.shape[0], device=self.device)

        enc = self.encoder(obs)                    # (B, H)
        lstm_in = enc.unsqueeze(1)                 # (B, 1, H)
        lstm_out, next_hidden = self.lstm(lstm_in, hidden)
        feat = lstm_out[:, -1, :]                  # (B, LSTM_H)

        mean = self.actor_mean(feat)              # (B, A)
        log_std = self.actor_log_std.expand_as(mean)
        value = self.critic(feat).squeeze(-1)      # (B,)
        return mean, log_std, value, next_hidden

    def forward_sequence(
        self,
        obs_seq: torch.Tensor,
        hidden: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    ):
        """
        obs_seq: (B, T, obs_dim)
        returns:
          mean: (B, T, action_dim)
          log_std: (B, T, action_dim)
          value: (B, T)
        """
        obs_seq = obs_seq.to(self.device)
        if obs_seq.dim() != 3:
            raise ValueError("obs_seq must have shape (B, T, obs_dim)")

        B, T, _ = obs_seq.shape
        if hidden is None:
            hidden = self.init_hidden(batch_size=B, device=self.device)

        enc = self.encoder(obs_seq.reshape(B * T, -1)).reshape(B, T, -1)
        lstm_out, _ = self.lstm(enc, hidden)
        mean = self.actor_mean(lstm_out)
        log_std = self.actor_log_std.view(1, 1, -1).expand_as(mean)
        value = self.critic(lstm_out).squeeze(-1)
        return mean, log_std, value

    def _squash_log_prob(self, raw_action: torch.Tensor, mean: torch.Tensor, std: torch.Tensor):
        dist = torch.distributions.Normal(mean, std)
        logp_raw = dist.log_prob(raw_action).sum(dim=-1)
        squashed = torch.tanh(raw_action)
        correction = torch.log(1.0 - squashed.pow(2) + 1e-6).sum(dim=-1)
        logp = logp_raw - correction
        entropy = dist.entropy().sum(dim=-1)
        return logp, entropy

    @torch.no_grad()
    def act_step(
        self,
        obs: np.ndarray | torch.Tensor,
        hidden: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        deterministic: bool = False,
    ):
        if not torch.is_tensor(obs):
            obs = torch.tensor(obs, dtype=torch.float32, device=self.device)
        obs = self._ensure_2d_obs(obs)

        mean, log_std, value, next_hidden = self.forward_step(obs, hidden)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mean, std)

        if deterministic:
            raw_action = mean
        else:
            raw_action = dist.sample()

        env_action = torch.tanh(raw_action)
        logp, _ = self._squash_log_prob(raw_action, mean, std)

        return (
            env_action.squeeze(0).detach().cpu().numpy(),
            float(logp.squeeze(0).item()),
            float(value.squeeze(0).item()),
            raw_action.squeeze(0).detach().cpu().numpy(),
            next_hidden,
        )

    def evaluate_actions(
        self,
        obs_seq: torch.Tensor,
        raw_action_seq: torch.Tensor,
        hidden: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    ):
        """
        obs_seq:       (B, T, obs_dim)
        raw_action_seq: (B, T, action_dim)  pre-tanh actions stored during rollout
        """
        obs_seq = obs_seq.to(self.device)
        raw_action_seq = raw_action_seq.to(self.device)

        mean, log_std, value = self.forward_sequence(obs_seq, hidden)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mean, std)

        logp_raw = dist.log_prob(raw_action_seq).sum(dim=-1)
        squashed = torch.tanh(raw_action_seq)
        correction = torch.log(1.0 - squashed.pow(2) + 1e-6).sum(dim=-1)
        logp = logp_raw - correction
        entropy = dist.entropy().sum(dim=-1)

        return logp, value, entropy

    def save(self, path: str):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save(self.state_dict(), path)

    def load(self, path: str):
        self.load_state_dict(torch.load(path, map_location=self.device))

In [ ]:
# ============================================================
# Cell 4 — Search environment
# ============================================================
class SearchEnv:
    """
    Search task environment.

    Observation (12):
      [uav_pos(2), uav_vel(2),
       vec_to_op_center(2), op_size(2),
       vec_to_search_center(2), search_size(2)]

    Action (3):
      [heading_x, heading_y, speed_raw] where each is interpreted in [-1, 1].
    """

    def __init__(
        self,
        op_min=(0.0, 0.0),
        op_max=(100.0, 100.0),
        search_min=(20.0, 20.0),
        search_max=(80.0, 80.0),
        grid_n: int = 6,
        dt: float = 1.0,
        min_speed: float = 5.0,
        max_speed: float = 20.0,
        max_steps: int = 200,
        coverage_goal: float = 0.60,
        start_in_search_prob: float = 0.70,
    ):
        self.op_min = np.array(op_min, dtype=np.float32)
        self.op_max = np.array(op_max, dtype=np.float32)
        self.search_min = np.array(search_min, dtype=np.float32)
        self.search_max = np.array(search_max, dtype=np.float32)

        self.grid_n = int(grid_n)
        self.dt = float(dt)
        self.min_speed = float(min_speed)
        self.max_speed = float(max_speed)
        self.max_steps = int(max_steps)
        self.coverage_goal = float(coverage_goal)
        self.start_in_search_prob = float(start_in_search_prob)

        self.obs_dim = 12
        self.act_dim = 3

        self.reset()

    def reset(self):
        if np.random.rand() < self.start_in_search_prob:
            self.pos = np.random.uniform(self.search_min, self.search_max).astype(np.float32)
        else:
            self.pos = np.random.uniform(self.op_min, self.op_max).astype(np.float32)

        self.vel = np.zeros(2, dtype=np.float32)
        self.visited = np.zeros((self.grid_n, self.grid_n), dtype=np.bool_)
        self.step_count = 0
        self.done = False
        self.success = False
        return self._obs()

    def _obs(self):
        op_center = rect_center(self.op_min, self.op_max)
        op_size = rect_size(self.op_min, self.op_max)
        search_center = rect_center(self.search_min, self.search_max)
        search_size = rect_size(self.search_min, self.search_max)

        return np.concatenate(
            [
                self.pos,
                self.vel,
                op_center - self.pos,
                op_size,
                search_center - self.pos,
                search_size,
            ]
        ).astype(np.float32)

    def _in_search(self, p):
        return rect_contains(p, self.search_min, self.search_max)

    def _in_op(self, p):
        return rect_contains(p, self.op_min, self.op_max)

    def _mark_visited(self):
        if not self._in_search(self.pos):
            return 0
        prev_count = int(self.visited.sum())
        rel = (self.pos - self.search_min) / np.maximum(self.search_max - self.search_min, 1e-6)
        rel = np.clip(rel, 0.0, 0.999999)
        ij = (rel * self.grid_n).astype(int)
        self.visited[ij[0], ij[1]] = True
        new_count = int(self.visited.sum())
        return max(0, new_count - prev_count)

    def coverage_fraction(self):
        return float(self.visited.mean())

    def step(self, action: np.ndarray):
        if self.done:
            raise RuntimeError("SearchEnv.step() called after done=True. Call reset().")

        action = np.asarray(action, dtype=np.float32)
        action = np.clip(action, -1.0, 1.0)

        heading = normalize(action[:2])
        if norm(heading) < 1e-8:
            heading = np.array([1.0, 0.0], dtype=np.float32)

        speed = self.min_speed + 0.5 * (action[2] + 1.0) * (self.max_speed - self.min_speed)

        prev_pos = self.pos.copy()
        prev_in_search = self._in_search(prev_pos)
        prev_in_op = self._in_op(prev_pos)

        next_raw = prev_pos + heading * speed * self.dt
        reward = -0.001

        if prev_in_op and not self._in_op(next_raw):
            reward -= 0.2

        self.pos = clip(next_raw, self.op_min, self.op_max).astype(np.float32)
        self.vel = ((self.pos - prev_pos) / max(self.dt, 1e-8)).astype(np.float32)

        in_search_after = self._in_search(self.pos)

        if (not prev_in_search) and in_search_after:
            reward += 0.1
        if prev_in_search and (not in_search_after):
            reward -= 0.2

        # Encourage constant exploration rather than hovering.
        if in_search_after:
            reward += 0.01
            reward += 0.001 * norm(self.pos - prev_pos)

        new_cells = self._mark_visited()
        reward += 0.05 * float(new_cells)

        coverage = self.coverage_fraction()
        if coverage >= self.coverage_goal and not self.done:
            reward += 1.0
            self.done = True
            self.success = True

        self.step_count += 1
        if self.step_count >= self.max_steps and not self.done:
            reward -= 0.5
            self.done = True

        info = {
            "coverage": coverage,
            "new_cells": int(new_cells),
            "success": self.success,
            "in_search": in_search_after,
            "in_op": self._in_op(self.pos),
        }
        return self._obs(), float(reward), self.done, info

In [ ]:
# ============================================================
# Cell 5 — Tracking environment
# ============================================================
class TrackingEnv:
    """
    Tracking task environment.

    Observation (13):
      [uav_pos(2), uav_vel(2),
       vec_to_op_center(2), op_size(2),
       vec_to_target_last_known_pos(2),
       target_last_known_vel(2),
       detected_bool(1)]

    Action (3):
      [heading_x, heading_y, speed_raw] where each is interpreted in [-1, 1].
    """

    def __init__(
        self,
        op_min=(0.0, 0.0),
        op_max=(100.0, 100.0),
        dt: float = 1.0,
        min_speed: float = 0.0,   # lowered so the policy can learn small corrective steps
        max_speed: float = 15.0,
        max_steps: int = 300,
        sensor_radius: float = 20.0,
        target_speed: float = 4.0,
        consecutive_detect_goal: int = 8,
        easy_start_prob: float = 0.60,
        easy_start_radius: float = 12.0,
    ):
        self.op_min = np.array(op_min, dtype=np.float32)
        self.op_max = np.array(op_max, dtype=np.float32)
        self.dt = float(dt)
        self.min_speed = float(min_speed)
        self.max_speed = float(max_speed)
        self.max_steps = int(max_steps)
        self.sensor_radius = float(sensor_radius)
        self.target_speed = float(target_speed)
        self.consecutive_detect_goal = int(consecutive_detect_goal)
        self.easy_start_prob = float(easy_start_prob)
        self.easy_start_radius = float(easy_start_radius)

        self.obs_dim = 13
        self.act_dim = 3

        self.reset()

    def _random_target_vel(self):
        theta = np.random.uniform(0.0, 2.0 * np.pi)
        return np.array([math.cos(theta), math.sin(theta)], dtype=np.float32) * self.target_speed

    def _generate_waypoints(self, n: int = 5):
        return [np.random.uniform(self.op_min, self.op_max).astype(np.float32) for _ in range(n)]

    def reset(self):
        self.target_pos = np.random.uniform(self.op_min, self.op_max).astype(np.float32)
        self.target_vel = self._random_target_vel()
        self.target_waypoints = self._generate_waypoints()

        if np.random.rand() < self.easy_start_prob:
            angle = np.random.uniform(0.0, 2.0 * np.pi)
            radius = np.random.uniform(0.0, self.easy_start_radius)
            offset = np.array([math.cos(angle), math.sin(angle)], dtype=np.float32) * radius
            self.pos = clip(self.target_pos + offset, self.op_min, self.op_max).astype(np.float32)
        else:
            self.pos = np.random.uniform(self.op_min, self.op_max).astype(np.float32)

        self.vel = np.zeros(2, dtype=np.float32)
        self.last_known_pos = self.target_pos.copy()
        self.last_known_vel = self.target_vel.copy()
        self.detected_prev = False
        self.consecutive_detect = 0
        self.step_count = 0
        self.done = False
        self.success = False
        return self._obs()

    def _move_target(self):
        if len(self.target_waypoints) == 0:
            self.target_waypoints = self._generate_waypoints()

        wp = self.target_waypoints[0]
        d = wp - self.target_pos
        dist = norm(d)

        if dist < max(self.target_speed * self.dt, 1e-6):
            self.target_pos = wp.copy()
            self.target_waypoints.pop(0)
            self.target_vel = self._random_target_vel()
        else:
            direction = normalize(d)
            self.target_vel = direction * self.target_speed
            self.target_pos = self.target_pos + self.target_vel * self.dt

        self.target_pos = clip(self.target_pos, self.op_min, self.op_max).astype(np.float32)

    def _detected(self):
        return norm(self.target_pos - self.pos) <= self.sensor_radius

    def _obs(self):
        op_center = rect_center(self.op_min, self.op_max)
        op_size = rect_size(self.op_min, self.op_max)
        detected = 1.0 if self._detected() else 0.0

        return np.concatenate(
            [
                self.pos,
                self.vel,
                op_center - self.pos,
                op_size,
                self.last_known_pos - self.pos,
                self.last_known_vel,
                np.array([detected], dtype=np.float32),
            ]
        ).astype(np.float32)

    def step(self, action: np.ndarray):
        if self.done:
            raise RuntimeError("TrackingEnv.step() called after done=True. Call reset().")

        action = np.asarray(action, dtype=np.float32)
        action = np.clip(action, -1.0, 1.0)

        heading = normalize(action[:2])
        if norm(heading) < 1e-8:
            heading = np.array([1.0, 0.0], dtype=np.float32)

        speed = self.min_speed + 0.5 * (action[2] + 1.0) * (self.max_speed - self.min_speed)

        prev_pos = self.pos.copy()
        prev_in_op = rect_contains(prev_pos, self.op_min, self.op_max)
        prev_dist = norm(prev_pos - self.last_known_pos)

        detected_before = self._detected()
        reward = -0.001

        if detected_before:
            reward += 0.05

        next_raw = prev_pos + heading * speed * self.dt
        if prev_in_op and not rect_contains(next_raw, self.op_min, self.op_max):
            reward -= 0.2

        self.pos = clip(next_raw, self.op_min, self.op_max).astype(np.float32)
        self.vel = ((self.pos - prev_pos) / max(self.dt, 1e-8)).astype(np.float32)

        new_dist = norm(self.pos - self.last_known_pos)
        reward += 0.02 * (prev_dist - new_dist)

        # Encourage smaller steps when the UAV is already close enough to be precise.
        # This helps reduce the step-size zig-zag around the target during tracking.
        if prev_dist <= 1.5 * self.sensor_radius:
            reward += 0.03 * (1.0 - (speed / max(self.max_speed, 1e-6)))

        self._move_target()

        detected_after = self._detected()
        if detected_after:
            self.last_known_pos = self.target_pos.copy()
            self.last_known_vel = self.target_vel.copy()
            reward += 0.1 if not detected_before else 0.05
            self.consecutive_detect += 1
        else:
            self.consecutive_detect = 0

        self.detected_prev = detected_after

        if self.consecutive_detect >= self.consecutive_detect_goal and not self.done:
            reward += 1.0
            self.done = True
            self.success = True

        self.step_count += 1
        if self.step_count >= self.max_steps and not self.done:
            reward -= 0.5
            self.done = True

        info = {
            "detected": detected_after,
            "success": self.success,
            "target_pos": self.target_pos.copy(),
            "target_vel": self.target_vel.copy(),
            "last_known_pos": self.last_known_pos.copy(),
            "last_known_vel": self.last_known_vel.copy(),
        }
        return self._obs(), float(reward), self.done, info

In [ ]:
# ============================================================
# Cell 6 — Mission data structures
# ============================================================
@dataclass
class UAVState:
    uav_id: int
    pos: np.ndarray
    vel: np.ndarray
    task_type: str = "search"   # search / track / idle
    assigned_area_id: Optional[int] = None
    tracked_target_id: Optional[int] = None


@dataclass
class GroundTarget:
    target_id: int
    pos: np.ndarray
    vel: np.ndarray
    waypoints: List[np.ndarray]
    active: bool = True
    detected: bool = False
    last_known_pos: Optional[np.ndarray] = None
    last_known_vel: Optional[np.ndarray] = None

    def step(self, op_min: np.ndarray, op_max: np.ndarray, dt: float, speed: float = 4.0):
        if not self.active:
            return

        if len(self.waypoints) == 0:
            self.waypoints = [np.random.uniform(op_min, op_max).astype(np.float32) for _ in range(5)]

        wp = self.waypoints[0]
        d = wp - self.pos
        dist = norm(d)

        if dist < max(speed * dt, 1e-6):
            self.pos = wp.copy()
            self.waypoints.pop(0)
            theta = np.random.uniform(0.0, 2.0 * np.pi)
            self.vel = np.array([math.cos(theta), math.sin(theta)], dtype=np.float32) * speed
        else:
            direction = normalize(d)
            self.vel = direction * speed
            self.pos = self.pos + self.vel * dt

        self.pos = clip(self.pos, op_min, op_max).astype(np.float32)
        self.last_known_pos = self.pos.copy()
        self.last_known_vel = self.vel.copy()

In [ ]:
# ============================================================
# Cell 7 — Central swarm controller
# ============================================================
class SwarmController:
    def __init__(
        self,
        uavs: List[UAVState],
        targets: List[GroundTarget],
        op_min: Tuple[float, float],
        op_max: Tuple[float, float],
        search_agent: PPORecurrentPolicy,
        tracking_agent: PPORecurrentPolicy,
        sensor_radius: float = 20.0,
        dt: float = 1.0,
    ):
        self.uavs = uavs
        self.targets = targets
        self.op_min = np.array(op_min, dtype=np.float32)
        self.op_max = np.array(op_max, dtype=np.float32)
        self.search_agent = search_agent
        self.tracking_agent = tracking_agent
        self.sensor_radius = float(sensor_radius)
        self.dt = float(dt)

        self.active_target_to_uav: Dict[int, int] = {}
        self.search_hidden: Dict[int, Tuple[torch.Tensor, torch.Tensor]] = {}
        self.track_hidden: Dict[int, Tuple[torch.Tensor, torch.Tensor]] = {}

        # Motion memory used to suppress frame-to-frame zig-zagging.
        self.prev_motion_dir: Dict[int, np.ndarray] = {}
        self.prev_motion_speed: Dict[int, float] = {}

        self.search_areas = self._partition_operation_area(len(self.uavs))
        self._assign_search_areas()

    def _partition_operation_area(self, n_parts: int):
        y_edges = np.linspace(self.op_min[1], self.op_max[1], n_parts + 1)
        areas = []
        for i in range(n_parts):
            mn = np.array([self.op_min[0], y_edges[i]], dtype=np.float32)
            mx = np.array([self.op_max[0], y_edges[i + 1]], dtype=np.float32)
            areas.append((mn, mx))
        return areas

    def _reset_search_hidden(self, uav_id: int):
        self.search_hidden[uav_id] = self.search_agent.init_hidden(batch_size=1, device=self.search_agent.device)

    def _reset_track_hidden(self, uav_id: int):
        self.track_hidden[uav_id] = self.tracking_agent.init_hidden(batch_size=1, device=self.tracking_agent.device)

    def _reset_motion_state(self, uav_id: int):
        self.prev_motion_dir[uav_id] = np.array([1.0, 0.0], dtype=np.float32)
        self.prev_motion_speed[uav_id] = 0.0

    def _assign_search_areas(self):
        search_uavs = [u for u in self.uavs if u.task_type != "track"]
        for idx, uav in enumerate(search_uavs):
            uav.task_type = "search"
            uav.assigned_area_id = idx if idx < len(self.search_areas) else None
            uav.tracked_target_id = None
            self._reset_search_hidden(uav.uav_id)
            self._reset_motion_state(uav.uav_id)

    def _search_obs(self, uav: UAVState, area_id: int):
        mn, mx = self.search_areas[area_id]
        op_center = rect_center(self.op_min, self.op_max)
        op_size = rect_size(self.op_min, self.op_max)
        area_center = rect_center(mn, mx)
        area_size = rect_size(mn, mx)

        return np.concatenate(
            [
                uav.pos,
                uav.vel,
                op_center - uav.pos,
                op_size,
                area_center - uav.pos,
                area_size,
            ]
        ).astype(np.float32)

    def _track_obs(self, uav: UAVState, target: GroundTarget):
        op_center = rect_center(self.op_min, self.op_max)
        op_size = rect_size(self.op_min, self.op_max)
        detected = 1.0 if norm(target.pos - uav.pos) <= self.sensor_radius else 0.0
        last_known_pos = target.last_known_pos if target.last_known_pos is not None else target.pos
        last_known_vel = target.last_known_vel if target.last_known_vel is not None else target.vel

        return np.concatenate(
            [
                uav.pos,
                uav.vel,
                op_center - uav.pos,
                op_size,
                last_known_pos - uav.pos,
                last_known_vel,
                np.array([detected], dtype=np.float32),
            ]
        ).astype(np.float32)

    def _smoothed_direction(self, uav_id: int, desired_dir: np.ndarray, task_type: str) -> np.ndarray:
        """
        Blend the newly proposed direction with the previous direction so the UAV
        keeps motion continuity instead of flipping every frame.
        """
        desired_dir = normalize(desired_dir)
        if norm(desired_dir) < 1e-8:
            desired_dir = self.prev_motion_dir.get(uav_id, np.array([1.0, 0.0], dtype=np.float32))

        prev_dir = self.prev_motion_dir.get(uav_id, np.array([1.0, 0.0], dtype=np.float32))

        # Stronger smoothing for tracking, slightly looser for search.
        alpha = 0.18 if task_type == "track" else 0.28

        blended = (1.0 - alpha) * prev_dir + alpha * desired_dir
        blended = normalize(blended)

        # If the new direction tries to reverse sharply, damp it further.
        # This suppresses the visible back-and-forth zig-zag artifact.
        if np.dot(prev_dir, blended) < -0.15:
            blended = normalize(0.85 * prev_dir + 0.15 * desired_dir)

        self.prev_motion_dir[uav_id] = blended
        return blended

    def _apply_action(
        self,
        uav: UAVState,
        action: np.ndarray,
        task_type: str = "search",
        target_distance: Optional[float] = None,
    ):
        action = np.asarray(action, dtype=np.float32)
        action = np.clip(action, -1.0, 1.0)

        desired_dir = normalize(action[:2])
        if norm(desired_dir) < 1e-8:
            desired_dir = self.prev_motion_dir.get(uav.uav_id, np.array([1.0, 0.0], dtype=np.float32))

        direction = self._smoothed_direction(uav.uav_id, desired_dir, task_type)

        # Search UAVs should keep moving broadly so they cover the area extensively.
        # Tracking UAVs should slow down as they approach the target to avoid overshoot.
        if task_type == "search":
            min_speed, max_speed = 5.0, 15.0
            speed = min_speed + 0.5 * (action[2] + 1.0) * (max_speed - min_speed)
        else:
            min_speed, max_speed = 0.0, 15.0
            speed = min_speed + 0.5 * (action[2] + 1.0) * (max_speed - min_speed)

            if target_distance is not None:
                # Reduce speed as the UAV gets close to the target.
                # This helps remove the step-toward / step-away oscillation.
                distance_scale = np.clip(target_distance / (2.0 * self.sensor_radius), 0.25, 1.0)
                speed *= float(distance_scale)

        prev = uav.pos.copy()
        nxt = prev + direction * speed * self.dt
        uav.pos = clip(nxt, self.op_min, self.op_max).astype(np.float32)
        delta = (uav.pos - prev).astype(np.float32)
        uav.vel = (delta / max(self.dt, 1e-8)).astype(np.float32)

        self.prev_motion_speed[uav.uav_id] = float(speed)
        return action, speed

    def _choose_tracker(self, target: GroundTarget, preferred_uav_id: Optional[int] = None):
        if preferred_uav_id is not None:
            return preferred_uav_id

        best_id = None
        best_d = float("inf")
        for uav in self.uavs:
            if uav.task_type != "track":
                d = norm(uav.pos - target.pos)
                if d < best_d:
                    best_d = d
                    best_id = uav.uav_id

        if best_id is not None:
            return best_id

        best_id = self.uavs[0].uav_id
        best_d = float("inf")
        for uav in self.uavs:
            d = norm(uav.pos - target.pos)
            if d < best_d:
                best_d = d
                best_id = uav.uav_id
        return best_id

    def _assign_tracking_task(self, target_id: int, tracker_id: int, task_logger: TableLogger, t: int):
        self.active_target_to_uav[target_id] = tracker_id
        tracker = self.uavs[tracker_id]
        tracker.task_type = "track"
        tracker.tracked_target_id = target_id
        tracker.assigned_area_id = None
        self._reset_track_hidden(tracker_id)
        self._reset_motion_state(tracker_id)

        task_logger.add(
            time=t,
            event="assign_tracking",
            target_id=target_id,
            tracker_uav_id=tracker_id,
        )

    def mission_summary(self):
        return {
            "active_target_to_uav": {str(k): int(v) for k, v in self.active_target_to_uav.items()},
            "search_areas": [
                {
                    "area_id": int(i),
                    "min": json_safe(mn),
                    "max": json_safe(mx),
                }
                for i, (mn, mx) in enumerate(self.search_areas)
            ],
            "uavs": [
                {
                    "uav_id": int(u.uav_id),
                    "pos": json_safe(u.pos),
                    "vel": json_safe(u.vel),
                    "task_type": str(u.task_type),
                    "assigned_area_id": None if u.assigned_area_id is None else int(u.assigned_area_id),
                    "tracked_target_id": None if u.tracked_target_id is None else int(u.tracked_target_id),
                }
                for u in self.uavs
            ],
            "targets": [
                {
                    "target_id": int(t.target_id),
                    "pos": json_safe(t.pos),
                    "vel": json_safe(t.vel),
                    "detected": bool(t.detected),
                    "last_known_pos": json_safe(t.last_known_pos),
                    "last_known_vel": json_safe(t.last_known_vel),
                }
                for t in self.targets
            ],
        }

    def step(self, t: int, mission_logger: TableLogger, task_logger: TableLogger):
        # Move all targets first.
        for target in self.targets:
            target.step(self.op_min, self.op_max, self.dt)

        detections: List[Tuple[int, int]] = []
        newly_assigned_trackers = set()

        # Only idle UAVs once all targets are already being tracked.
        all_targets_tracked = len(self.active_target_to_uav) >= len(self.targets)

        # SEARCH / IDLE PHASE
        for uav in self.uavs:
            if all_targets_tracked and uav.task_type != "track":
                uav.task_type = "idle"
                uav.assigned_area_id = None
                uav.tracked_target_id = None
                uav.vel[:] = 0.0

                mission_logger.add(
                    time=t,
                    uav_id=uav.uav_id,
                    task_type="idle",
                    assigned_area_id=None,
                    tracked_target_id=None,
                    uav_x=float(uav.pos[0]),
                    uav_y=float(uav.pos[1]),
                    uav_vx=0.0,
                    uav_vy=0.0,
                    action_x=0.0,
                    action_y=0.0,
                    action_speed=0.0,
                    raw_action_x=0.0,
                    raw_action_y=0.0,
                    raw_action_speed=0.0,
                    reward=0.0,
                    detected=False,
                    target_id=None,
                )
                continue

            if uav.task_type != "search":
                continue

            if uav.assigned_area_id is None:
                action_env = np.array([0.0, 0.0, 0.0], dtype=np.float32)
                action_raw = np.array([0.0, 0.0, 0.0], dtype=np.float32)
                reward = 0.0
                task_type = "idle"
                detected = False
                detected_target_id = None
            else:
                obs = self._search_obs(uav, uav.assigned_area_id)
                hidden = self.search_hidden.get(uav.uav_id)
                if hidden is None:
                    self._reset_search_hidden(uav.uav_id)
                    hidden = self.search_hidden[uav.uav_id]

                action_env, logp, value, action_raw, next_hidden = self.search_agent.act_step(
                    obs, hidden, deterministic=True
                )
                self.search_hidden[uav.uav_id] = next_hidden
                self._apply_action(uav, action_env, task_type="search")

                reward = 0.0
                task_type = "search"
                detected = False
                detected_target_id = None

                # Detect the first target in sensor range.
                for target in self.targets:
                    if norm(target.pos - uav.pos) <= self.sensor_radius:
                        detected = True
                        detected_target_id = target.target_id
                        target.detected = True
                        target.last_known_pos = target.pos.copy()
                        target.last_known_vel = target.vel.copy()
                        detections.append((target.target_id, uav.uav_id))
                        break

            mission_logger.add(
                time=t,
                uav_id=uav.uav_id,
                task_type=task_type,
                assigned_area_id=None if uav.assigned_area_id is None else int(uav.assigned_area_id),
                tracked_target_id=None if uav.tracked_target_id is None else int(uav.tracked_target_id),
                uav_x=float(uav.pos[0]),
                uav_y=float(uav.pos[1]),
                uav_vx=float(uav.vel[0]),
                uav_vy=float(uav.vel[1]),
                action_x=float(action_env[0]),
                action_y=float(action_env[1]),
                action_speed=float(action_env[2]),
                raw_action_x=float(action_raw[0]),
                raw_action_y=float(action_raw[1]),
                raw_action_speed=float(action_raw[2]),
                reward=float(reward),
                detected=bool(detected),
                target_id=detected_target_id,
            )

            if detected:
                task_logger.add(
                    time=t,
                    event="target_detected",
                    target_id=detected_target_id,
                    detector_uav_id=uav.uav_id,
                )

        # Assign new tracking tasks after search phase.
        for target_id, detector_uav_id in detections:
            if target_id not in self.active_target_to_uav:
                target = self.targets[target_id]
                tracker_id = self._choose_tracker(target, preferred_uav_id=detector_uav_id)
                self._assign_tracking_task(target_id, tracker_id, task_logger, t)
                newly_assigned_trackers.add(tracker_id)
                self._assign_search_areas()
                task_logger.add(
                    time=t,
                    event="reassign_search_areas",
                    reason="tracking_assigned",
                    target_id=target_id,
                )

        # TRACKING PHASE
        for uav in self.uavs:
            if uav.task_type != "track":
                continue
            if uav.uav_id in newly_assigned_trackers:
                continue
            if uav.tracked_target_id is None:
                continue

            target = self.targets[uav.tracked_target_id]
            obs = self._track_obs(uav, target)
            hidden = self.track_hidden.get(uav.uav_id)
            if hidden is None:
                self._reset_track_hidden(uav.uav_id)
                hidden = self.track_hidden[uav.uav_id]

            target_distance = norm(target.pos - uav.pos)
            action_env, logp, value, action_raw, next_hidden = self.tracking_agent.act_step(
                obs, hidden, deterministic=True
            )
            self.track_hidden[uav.uav_id] = next_hidden
            self._apply_action(uav, action_env, task_type="track", target_distance=target_distance)

            detected = norm(target.pos - uav.pos) <= self.sensor_radius
            if detected:
                target.detected = True
                target.last_known_pos = target.pos.copy()
                target.last_known_vel = target.vel.copy()

            mission_logger.add(
                time=t,
                uav_id=uav.uav_id,
                task_type="track",
                assigned_area_id=None,
                tracked_target_id=int(uav.tracked_target_id),
                uav_x=float(uav.pos[0]),
                uav_y=float(uav.pos[1]),
                uav_vx=float(uav.vel[0]),
                uav_vy=float(uav.vel[1]),
                action_x=float(action_env[0]),
                action_y=float(action_env[1]),
                action_speed=float(action_env[2]),
                raw_action_x=float(action_raw[0]),
                raw_action_y=float(action_raw[1]),
                raw_action_speed=float(action_raw[2]),
                reward=0.0,
                detected=bool(detected),
                target_id=int(target.target_id),
            )

            if detected:
                task_logger.add(
                    time=t,
                    event="tracking_update",
                    target_id=target.target_id,
                    tracker_uav_id=uav.uav_id,
                )

In [ ]:
# ============================================================
# Cell 8 — PPO + LSTM training loop
# ============================================================
def compute_gae(
    values: torch.Tensor,
    rewards: torch.Tensor,
    dones: torch.Tensor,
    gamma: float = 0.99,
    lam: float = 0.95,
):
    """
    values:  (T+1,)
    rewards: (T,)
    dones:   (T,)
    """
    T = rewards.shape[0]
    adv = torch.zeros(T, device=values.device)
    gae = 0.0

    for t in reversed(range(T)):
        mask = 1.0 - dones[t]
        delta = rewards[t] + gamma * values[t + 1] * mask - values[t]
        gae = delta + gamma * lam * mask * gae
        adv[t] = gae

    returns = adv + values[:-1]
    return adv, returns


class PPORecurrentTrainer:
    def __init__(
        self,
        policy: PPORecurrentPolicy,
        lr: float = 3e-4,
        clip_eps: float = 0.2,
        gamma: float = 0.99,
        lam: float = 0.95,
        ppo_epochs: int = 6,
        vf_coef: float = 0.5,
        entropy_coef: float = 0.01,
        max_grad_norm: float = 0.5,
    ):
        self.policy = policy
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.clip_eps = float(clip_eps)
        self.gamma = float(gamma)
        self.lam = float(lam)
        self.ppo_epochs = int(ppo_epochs)
        self.vf_coef = float(vf_coef)
        self.entropy_coef = float(entropy_coef)
        self.max_grad_norm = float(max_grad_norm)
        self.device = next(self.policy.parameters()).device

    def collect_episodes(
        self,
        env,
        min_steps: int,
        tag: str,
        episode_logger: Optional[TableLogger] = None,
        step_logger: Optional[TableLogger] = None,
        episode_offset: int = 0,
        deterministic: bool = False,
    ):
        trajectories = []
        total_steps = 0
        episode_idx = episode_offset

        while total_steps < min_steps:
            obs = env.reset()
            hidden = self.policy.init_hidden(batch_size=1, device=self.device)

            traj = {
                "obs": [],
                "raw_actions": [],
                "env_actions": [],
                "logp": [],
                "values": [],
                "rewards": [],
                "dones": [],
                "episode_reward": 0.0,
                "steps": 0,
                "success": False,
                "final_info": {},
            }

            done = False
            step_idx = 0
            last_info = {}

            while not done:
                obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device)
                with torch.no_grad():
                    action_env, logp, value, action_raw, hidden = self.policy.act_step(
                        obs_t, hidden, deterministic=deterministic
                    )

                next_obs, reward, done, info = env.step(action_env)

                traj["obs"].append(np.asarray(obs, dtype=np.float32))
                traj["raw_actions"].append(np.asarray(action_raw, dtype=np.float32))
                traj["env_actions"].append(np.asarray(action_env, dtype=np.float32))
                traj["logp"].append(float(logp))
                traj["values"].append(float(value))
                traj["rewards"].append(float(reward))
                traj["dones"].append(float(done))
                traj["episode_reward"] += float(reward)
                traj["steps"] += 1
                last_info = info

                if step_logger is not None:
                    row = {
                        "mode": tag,
                        "episode": episode_idx,
                        "step": step_idx,
                        "reward": float(reward),
                        "done": bool(done),
                        "uav_x": float(env.pos[0]),
                        "uav_y": float(env.pos[1]),
                        "uav_vx": float(env.vel[0]),
                        "uav_vy": float(env.vel[1]),
                        "action_x": float(action_env[0]),
                        "action_y": float(action_env[1]),
                        "action_speed": float(action_env[2]),
                        "raw_action_x": float(action_raw[0]),
                        "raw_action_y": float(action_raw[1]),
                        "raw_action_speed": float(action_raw[2]),
                    }

                    if isinstance(env, SearchEnv):
                        row["coverage"] = info.get("coverage", np.nan)
                        row["new_cells"] = info.get("new_cells", np.nan)
                        row["success"] = info.get("success", False)
                        row["in_search"] = info.get("in_search", False)
                        row["in_op"] = info.get("in_op", False)
                    elif isinstance(env, TrackingEnv):
                        row["detected"] = info.get("detected", False)
                        row["success"] = info.get("success", False)
                        tp = info.get("target_pos", np.array([np.nan, np.nan]))
                        tv = info.get("target_vel", np.array([np.nan, np.nan]))
                        lkp = info.get("last_known_pos", np.array([np.nan, np.nan]))
                        lkv = info.get("last_known_vel", np.array([np.nan, np.nan]))
                        row["target_x"] = float(tp[0])
                        row["target_y"] = float(tp[1])
                        row["target_vx"] = float(tv[0])
                        row["target_vy"] = float(tv[1])
                        row["last_known_x"] = float(lkp[0])
                        row["last_known_y"] = float(lkp[1])
                        row["last_known_vx"] = float(lkv[0])
                        row["last_known_vy"] = float(lkv[1])

                    step_logger.add(**row)

                obs = next_obs
                step_idx += 1
                total_steps += 1

            traj["success"] = bool(last_info.get("success", False))
            traj["final_info"] = json_safe(last_info)
            trajectories.append(traj)

            if episode_logger is not None:
                episode_logger.add(
                    mode=tag,
                    episode=episode_idx,
                    total_reward=traj["episode_reward"],
                    steps=traj["steps"],
                    success=traj["success"],
                    final_info=traj["final_info"],
                )

            episode_idx += 1

        return trajectories, episode_idx

    def ppo_update(self, trajectories: List[Dict[str, Any]]):
        # Compute advantages / returns for every episode first.
        advantages_all = []
        for traj in trajectories:
            rewards = torch.tensor(traj["rewards"], dtype=torch.float32, device=self.device)
            dones = torch.tensor(traj["dones"], dtype=torch.float32, device=self.device)
            values = torch.tensor(traj["values"] + [0.0], dtype=torch.float32, device=self.device)

            adv, ret = compute_gae(values, rewards, dones, gamma=self.gamma, lam=self.lam)
            traj["advantages"] = adv
            traj["returns"] = ret
            advantages_all.append(adv)

        all_adv = torch.cat([a.reshape(-1) for a in advantages_all], dim=0)
        adv_mean = all_adv.mean()
        adv_std = all_adv.std() + 1e-8

        for traj in trajectories:
            traj["advantages"] = (traj["advantages"] - adv_mean) / adv_std

        losses = []

        for _ in range(self.ppo_epochs):
            random.shuffle(trajectories)

            for traj in trajectories:
                obs = torch.tensor(np.asarray(traj["obs"]), dtype=torch.float32, device=self.device).unsqueeze(0)
                raw_actions = torch.tensor(np.asarray(traj["raw_actions"]), dtype=torch.float32, device=self.device).unsqueeze(0)
                old_logp = torch.tensor(np.asarray(traj["logp"]), dtype=torch.float32, device=self.device).unsqueeze(0)
                adv = traj["advantages"].unsqueeze(0)
                ret = traj["returns"].unsqueeze(0)

                logp, values, entropy = self.policy.evaluate_actions(obs, raw_actions)

                ratio = torch.exp(logp - old_logp)
                surr1 = ratio * adv
                surr2 = torch.clamp(ratio, 1.0 - self.clip_eps, 1.0 + self.clip_eps) * adv

                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss = ((values - ret) ** 2).mean()
                entropy_bonus = entropy.mean()

                loss = actor_loss + self.vf_coef * critic_loss - self.entropy_coef * entropy_bonus

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
                self.optimizer.step()

                losses.append(float(loss.item()))

        return float(np.mean(losses)) if losses else 0.0

    def train(
        self,
        env,
        iterations: int,
        rollout_steps: int,
        tag: str,
        episode_logger: Optional[TableLogger] = None,
        step_logger: Optional[TableLogger] = None,
        print_every: int = 25,
        deterministic_rollout: bool = False,
    ):
        episode_idx = 0
        reward_history = []
        success_history = []
        last_loss = 0.0

        for it in range(iterations):
            trajectories, episode_idx = self.collect_episodes(
                env=env,
                min_steps=rollout_steps,
                tag=tag,
                episode_logger=episode_logger,
                step_logger=step_logger,
                episode_offset=episode_idx,
                deterministic=deterministic_rollout,
            )

            last_loss = self.ppo_update(trajectories)

            ep_rewards = [traj["episode_reward"] for traj in trajectories]
            ep_success = [1.0 if traj["success"] else 0.0 for traj in trajectories]
            reward_history.extend(ep_rewards)
            success_history.extend(ep_success)

            if (it + 1) % print_every == 0 or it == 0 or it == iterations - 1:
                recent_rewards = reward_history[-print_every:] if len(reward_history) >= print_every else reward_history
                recent_success = success_history[-print_every:] if len(success_history) >= print_every else success_history

                avg_reward = float(np.mean(recent_rewards)) if recent_rewards else 0.0
                success_rate = float(np.mean(recent_success)) if recent_success else 0.0

                print(
                    f"[{tag}] epoch {it + 1}/{iterations}: "
                    f"avg_reward={avg_reward:.3f} | "
                    f"success_rate={success_rate:.2f} | "
                    f"last_loss={last_loss:.4f}"
                )

        return {
            "last_loss": last_loss,
            "avg_reward": float(np.mean(reward_history)) if reward_history else 0.0,
            "success_rate": float(np.mean(success_history)) if success_history else 0.0,
        }

In [ ]:
# ============================================================
# Cell 9 — Example mission generation
# ============================================================
def build_example_mission(
    n_uavs: int = 3,
    n_targets: int = 1,
    op_min=(0.0, 0.0),
    op_max=(100.0, 100.0),
):
    op_min = np.array(op_min, dtype=np.float32)
    op_max = np.array(op_max, dtype=np.float32)

    # Spread UAVs along the lower edge for a clear rollout.
    uavs = []
    x_positions = np.linspace(20.0, 80.0, n_uavs)
    for i in range(n_uavs):
        uavs.append(
            UAVState(
                uav_id=i,
                pos=np.array([float(x_positions[i]), 5.0], dtype=np.float32),
                vel=np.zeros(2, dtype=np.float32),
                task_type="search",
                assigned_area_id=i,
                tracked_target_id=None,
            )
        )

    # Use stable, visually meaningful target locations.
    preset_targets = [
        np.array([50.0, 35.0], dtype=np.float32),
        np.array([30.0, 65.0], dtype=np.float32),
        np.array([70.0, 65.0], dtype=np.float32),
    ]

    targets = []
    for tid in range(n_targets):
        if tid < len(preset_targets):
            pos = clip(preset_targets[tid], op_min, op_max).astype(np.float32)
        else:
            pos = np.random.uniform(op_min, op_max).astype(np.float32)

        theta = np.random.uniform(0.0, 2.0 * np.pi)
        vel = np.array([math.cos(theta), math.sin(theta)], dtype=np.float32) * 4.0
        waypoints = [np.random.uniform(op_min, op_max).astype(np.float32) for _ in range(6)]

        targets.append(
            GroundTarget(
                target_id=tid,
                pos=pos,
                vel=vel,
                waypoints=waypoints,
                active=True,
                detected=False,
                last_known_pos=pos.copy(),
                last_known_vel=vel.copy(),
            )
        )

    return uavs, targets

In [ ]:
# ============================================================
# Cell 10 — Example swarm rollout and ordered CSV exports
# ============================================================
def run_example_swarm_mission(
    search_agent: PPORecurrentPolicy,
    tracking_agent: PPORecurrentPolicy,
    out_dir: str,
    max_steps: int = 250,
    n_uavs: int = 3,
    n_targets: int = 1,
):
    os.makedirs(out_dir, exist_ok=True)

    uavs, targets = build_example_mission(n_uavs=n_uavs, n_targets=n_targets)

    controller = SwarmController(
        uavs=uavs,
        targets=targets,
        op_min=(0.0, 0.0),
        op_max=(100.0, 100.0),
        search_agent=search_agent,
        tracking_agent=tracking_agent,
        sensor_radius=20.0,
        dt=1.0,
    )

    mission_log = TableLogger()
    task_log = TableLogger()
    target_log = TableLogger()

    # Initial state snapshot before the first step.
    for uav in controller.uavs:
        mission_log.add(
            time=-1,
            uav_id=uav.uav_id,
            task_type=uav.task_type,
            assigned_area_id=None if uav.assigned_area_id is None else int(uav.assigned_area_id),
            tracked_target_id=None if uav.tracked_target_id is None else int(uav.tracked_target_id),
            uav_x=float(uav.pos[0]),
            uav_y=float(uav.pos[1]),
            uav_vx=float(uav.vel[0]),
            uav_vy=float(uav.vel[1]),
            action_x=0.0,
            action_y=0.0,
            action_speed=0.0,
            raw_action_x=0.0,
            raw_action_y=0.0,
            raw_action_speed=0.0,
            reward=0.0,
            detected=False,
            target_id=None,
        )
        task_log.add(
            time=-1,
            event="initial_search_assignment",
            uav_id=uav.uav_id,
            area_id=None if uav.assigned_area_id is None else int(uav.assigned_area_id),
        )

    for target in controller.targets:
        target_log.add(
            time=-1,
            target_id=target.target_id,
            target_x=float(target.pos[0]),
            target_y=float(target.pos[1]),
            target_vx=float(target.vel[0]),
            target_vy=float(target.vel[1]),
            target_detected=bool(target.detected),
            target_last_known_x=float(target.last_known_pos[0]),
            target_last_known_y=float(target.last_known_pos[1]),
            target_last_known_vx=float(target.last_known_vel[0]),
            target_last_known_vy=float(target.last_known_vel[1]),
        )

    # Rollout.
    for t in range(max_steps):
        controller.step(t=t, mission_logger=mission_log, task_logger=task_log)

        for target in controller.targets:
            target_log.add(
                time=t,
                target_id=target.target_id,
                target_x=float(target.pos[0]),
                target_y=float(target.pos[1]),
                target_vx=float(target.vel[0]),
                target_vy=float(target.vel[1]),
                target_detected=bool(target.detected),
                target_last_known_x=float(target.last_known_pos[0]),
                target_last_known_y=float(target.last_known_pos[1]),
                target_last_known_vx=float(target.last_known_vel[0]),
                target_last_known_vy=float(target.last_known_vel[1]),
            )

    # Save logs.
    mission_log.save_csv(os.path.join(out_dir, "mission_rollout.csv"))
    task_log.save_csv(os.path.join(out_dir, "task_events.csv"))
    target_log.save_csv(os.path.join(out_dir, "target_states.csv"))

    with open(os.path.join(out_dir, "mission_summary.json"), "w", encoding="utf-8") as f:
        json.dump(json_safe(controller.mission_summary()), f, indent=2)

    return controller

In [ ]:
# ============================================================
# Cell 11 — Full pipeline: train search, train tracking, rollout
# ============================================================
def main(
    output_dir: str = "/content/swarm_artifacts",
    seed: int = 42,
    search_iterations: int = 250,
    tracking_iterations: int = 350,
    rollout_steps: int = 1024,
    n_uavs: int = 3,
    n_targets: int = 1,
):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    # --------------------------------------------------------
    # Train search policy
    # --------------------------------------------------------
    print("Starting search-policy training...")
    search_env = SearchEnv(
        op_min=(0.0, 0.0),
        op_max=(100.0, 100.0),
        search_min=(20.0, 20.0),
        search_max=(80.0, 80.0),
        grid_n=6,
        dt=1.0,
        min_speed=5.0,
        max_speed=20.0,
        max_steps=200,
        coverage_goal=0.60,
        start_in_search_prob=0.70,
    )

    search_agent = PPORecurrentPolicy(
        obs_dim=search_env.obs_dim,
        action_dim=search_env.act_dim,
        hidden_size=256,
        lstm_hidden_size=256,
        lstm_layers=1,
    )

    search_trainer = PPORecurrentTrainer(
        policy=search_agent,
        lr=3e-4,
        clip_eps=0.2,
        gamma=0.99,
        lam=0.95,
        ppo_epochs=6,
        vf_coef=0.5,
        entropy_coef=0.01,
        max_grad_norm=0.5,
    )

    search_ep_log = TableLogger()
    search_step_log = TableLogger()

    search_stats = search_trainer.train(
        env=search_env,
        iterations=search_iterations,
        rollout_steps=rollout_steps,
        tag="search",
        episode_logger=search_ep_log,
        step_logger=search_step_log,
        print_every=25,
        deterministic_rollout=False,
    )

    search_agent.save(os.path.join(output_dir, "search_policy.pt"))
    search_ep_log.save_csv(os.path.join(output_dir, "search_training_episodes.csv"))
    search_step_log.save_csv(os.path.join(output_dir, "search_training_steps.csv"))

    # --------------------------------------------------------
    # Train tracking policy
    # --------------------------------------------------------
    print("Starting tracking-policy training...")
    tracking_env = TrackingEnv(
        op_min=(0.0, 0.0),
        op_max=(100.0, 100.0),
        dt=1.0,
        min_speed=5.0,
        max_speed=20.0,
        max_steps=300,
        sensor_radius=20.0,
        target_speed=4.0,
        consecutive_detect_goal=8,
        easy_start_prob=0.60,
        easy_start_radius=12.0,
    )

    tracking_agent = PPORecurrentPolicy(
        obs_dim=tracking_env.obs_dim,
        action_dim=tracking_env.act_dim,
        hidden_size=256,
        lstm_hidden_size=256,
        lstm_layers=1,
    )

    tracking_trainer = PPORecurrentTrainer(
        policy=tracking_agent,
        lr=3e-4,
        clip_eps=0.2,
        gamma=0.99,
        lam=0.95,
        ppo_epochs=6,
        vf_coef=0.5,
        entropy_coef=0.01,
        max_grad_norm=0.5,
    )

    tracking_ep_log = TableLogger()
    tracking_step_log = TableLogger()

    tracking_stats = tracking_trainer.train(
        env=tracking_env,
        iterations=tracking_iterations,
        rollout_steps=rollout_steps,
        tag="tracking",
        episode_logger=tracking_ep_log,
        step_logger=tracking_step_log,
        print_every=25,
        deterministic_rollout=False,
    )

    tracking_agent.save(os.path.join(output_dir, "tracking_policy.pt"))
    tracking_ep_log.save_csv(os.path.join(output_dir, "tracking_training_episodes.csv"))
    tracking_step_log.save_csv(os.path.join(output_dir, "tracking_training_steps.csv"))

    # --------------------------------------------------------
    # Run one example mission rollout with logs
    # --------------------------------------------------------
    print("Running example swarm rollout...")
    controller = run_example_swarm_mission(
        search_agent=search_agent,
        tracking_agent=tracking_agent,
        out_dir=output_dir,
        max_steps=250,
        n_uavs=n_uavs,
        n_targets=n_targets,
    )

    # --------------------------------------------------------
    # Save run config separately as JSON-safe data
    # --------------------------------------------------------
    run_config = {
        "seed": seed,
        "search_iterations": search_iterations,
        "tracking_iterations": tracking_iterations,
        "rollout_steps": rollout_steps,
        "n_uavs": n_uavs,
        "n_targets": n_targets,
        "output_dir": output_dir,
        "search_stats": search_stats,
        "tracking_stats": tracking_stats,
        "notes": "PPO + LSTM search and tracking policies; obstacle avoidance omitted.",
    }

    with open(os.path.join(output_dir, "run_config.json"), "w", encoding="utf-8") as f:
        json.dump(json_safe(run_config), f, indent=2)

    print("Done.")
    print(f"Artifacts saved in: {output_dir}")
    return controller


# In Colab, run this after all cells above:
controller = main(
    output_dir="/content/swarm_artifacts",
    seed=42,
    search_iterations=25,
    tracking_iterations=35,
    rollout_steps=1024,
    n_uavs=3,
    n_targets=3,
)

Starting search-policy training...
[search] epoch 1/25: avg_reward=-17.746 | success_rate=0.00 | last_loss=0.1654
[search] epoch 25/25: avg_reward=4.375 | success_rate=0.84 | last_loss=-0.0411
Starting tracking-policy training...
[tracking] epoch 1/35: avg_reward=-28.926 | success_rate=0.25 | last_loss=0.1985
[tracking] epoch 25/35: avg_reward=5.039 | success_rate=0.96 | last_loss=0.2249
[tracking] epoch 35/35: avg_reward=5.140 | success_rate=1.00 | last_loss=0.5001
Running example swarm rollout...
Done.
Artifacts saved in: /content/swarm_artifacts


In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from matplotlib.patches import Rectangle
from tqdm import tqdm   # <-- NEW

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
OUTPUT_DIR = "/content/swarm_artifacts"
MISSION_CSV = os.path.join(OUTPUT_DIR, "mission_rollout.csv")
TARGET_CSV = os.path.join(OUTPUT_DIR, "target_states.csv")
CONFIG_JSON = os.path.join(OUTPUT_DIR, "run_config.json")
SUMMARY_JSON = os.path.join(OUTPUT_DIR, "mission_summary.json")

OUT_MP4 = os.path.join(OUTPUT_DIR, "swarm_rollout.mp4")
OUT_GIF = os.path.join(OUTPUT_DIR, "swarm_rollout.gif")

TRAIL_LENGTH = 5   # <-- NEW: number of visible trail frames
DETECTION_AVG_WINDOW = 40

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def load_json(path):
    if not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def safe_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def infer_bounds(mission_df, target_df=None):
    xs, ys = [], []
    if "uav_x" in mission_df.columns:
        xs.append(mission_df["uav_x"].astype(float).values)
        ys.append(mission_df["uav_y"].astype(float).values)
    if target_df is not None and "target_x" in target_df.columns:
        xs.append(target_df["target_x"].astype(float).values)
        ys.append(target_df["target_y"].astype(float).values)

    xs = np.concatenate(xs) if xs else np.array([0, 100])
    ys = np.concatenate(ys) if ys else np.array([0, 100])

    xmin, xmax = float(np.nanmin(xs)), float(np.nanmax(xs))
    ymin, ymax = float(np.nanmin(ys)), float(np.nanmax(ys))

    pad_x = max(5.0, 0.08 * (xmax - xmin + 1e-9))
    pad_y = max(5.0, 0.08 * (ymax - ymin + 1e-9))
    return (xmin - pad_x, xmax + pad_x), (ymin - pad_y, ymax + pad_y)

def build_search_areas(n_uavs, xlim, ylim):
    ymin, ymax = ylim
    strips = []
    edges = np.linspace(ymin, ymax, n_uavs + 1)
    for i in range(n_uavs):
        strips.append((xlim[0], edges[i], xlim[1] - xlim[0], edges[i + 1] - edges[i]))
    return strips

def get_search_area_color(area_id):
    colors = [
        "#ffd6a5", "#fdffb6", "#caffbf", "#9bf6ff",
        "#a0c4ff", "#bdb2ff", "#ffc6ff", "#ffadad"
    ]
    return colors[area_id % len(colors)]

# ------------------------------------------------------------
# Load stored rollout data
# ------------------------------------------------------------
mission_df = pd.read_csv(MISSION_CSV)
target_df = pd.read_csv(TARGET_CSV) if os.path.exists(TARGET_CSV) else pd.DataFrame()
config = load_json(CONFIG_JSON)
summary = load_json(SUMMARY_JSON)

mission_df = safe_numeric(
    mission_df,
    ["time", "uav_id", "uav_x", "uav_y", "uav_vx", "uav_vy",
     "action_x", "action_y", "action_speed",
     "reward", "target_id", "assigned_area_id"]
)
if not target_df.empty:
    target_df = safe_numeric(
        target_df,
        ["time", "target_id", "target_x", "target_y", "target_vx", "target_vy",
         "target_last_known_x", "target_last_known_y",
         "target_last_known_vx", "target_last_known_vy"]
    )

mission_df = mission_df.sort_values(["time", "uav_id"]).reset_index(drop=True)
if not target_df.empty:
    target_df = target_df.sort_values(["time", "target_id"]).reset_index(drop=True)

times = sorted(mission_df["time"].dropna().unique().tolist())

# ------------------------------------------------------------
# Infer bounds, areas, IDs
# ------------------------------------------------------------
xlim, ylim = infer_bounds(mission_df, target_df if not target_df.empty else None)
n_uavs = int(config.get("n_uavs", mission_df["uav_id"].nunique()))
search_areas = build_search_areas(n_uavs, xlim, ylim)

uav_ids = sorted(mission_df["uav_id"].dropna().unique().astype(int).tolist())
uav_colors = {uid: plt.cm.tab10(i % 10) for i, uid in enumerate(uav_ids)}

target_ids = []
if not target_df.empty and "target_id" in target_df.columns:
    target_ids = sorted(target_df["target_id"].dropna().unique().astype(int).tolist())
elif "target_id" in mission_df.columns:
    target_ids = sorted(mission_df["target_id"].dropna().unique().astype(int).tolist())

target_colors = {tid: plt.cm.Dark2(i % 8) for i, tid in enumerate(target_ids)}

mission_by_time = {t: g.copy() for t, g in mission_df.groupby("time")}
target_by_time = {t: g.copy() for t, g in target_df.groupby("time")} if not target_df.empty else {}

# ------------------------------------------------------------
# Figure setup
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect("equal")
ax.set_title("Swarm rollout replay from stored logs")
ax.set_xlabel("X")
ax.set_ylabel("Y")

# Search areas
for i, (x, y, w, h) in enumerate(search_areas):
    ax.add_patch(Rectangle((x, y), w, h, fill=True, alpha=0.18,
                           facecolor=get_search_area_color(i),
                           edgecolor="gray", linewidth=1.0))

# Artists
uav_scatter = ax.scatter([], [], s=80, marker="o")
uav_trails = {uid: ax.plot([], [], linewidth=1.4, alpha=0.9)[0] for uid in uav_ids}
uav_texts = []

target_scatter = ax.scatter([], [], s=90, marker="X")
target_trails = {tid: ax.plot([], [], linestyle="--", linewidth=1.2, alpha=0.9)[0] for tid in target_ids}
target_texts = []

time_text = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top", fontsize=12, weight="bold")
info_text = ax.text(0.02, 0.94, "", transform=ax.transAxes, va="top", fontsize=10)

# NEW: histories limited to TRAIL_LENGTH
uav_history = {uid: {"x": [], "y": []} for uid in uav_ids}
target_history = {tid: {"x": [], "y": []} for tid in target_ids}

# NEW: cumulative detection count
cumulative_detections = 0
recent_detections = []

def get_frame_state(t):
    m = mission_by_time.get(t, pd.DataFrame())
    tg = target_by_time.get(t, pd.DataFrame()) if target_by_time else pd.DataFrame()
    return m, tg

def update(frame_idx):
    global cumulative_detections

    t = times[frame_idx]
    m, tg = get_frame_state(t)

    # UAVs
    uav_xy = []
    uav_col = []

    for txt in uav_texts:
        txt.remove()
    uav_texts.clear()

    for _, row in m.iterrows():
        uid = int(row["uav_id"])
        x, y = float(row["uav_x"]), float(row["uav_y"])
        task = str(row.get("task_type", ""))
        area_id = row.get("assigned_area_id", np.nan)

        # --- NEW: sliding window trail ---
        uav_history[uid]["x"].append(x)
        uav_history[uid]["y"].append(y)
        uav_history[uid]["x"] = uav_history[uid]["x"][-TRAIL_LENGTH:]
        uav_history[uid]["y"] = uav_history[uid]["y"][-TRAIL_LENGTH:]

        uav_xy.append([x, y])
        uav_col.append(uav_colors[uid])

        uav_trails[uid].set_data(uav_history[uid]["x"], uav_history[uid]["y"])

        label = f"U{uid}:{task}"
        if pd.notna(area_id):
            label += f" A{int(area_id)}"
        txt = ax.text(x + 0.6, y + 0.6, label, fontsize=9)
        uav_texts.append(txt)

    uav_scatter.set_offsets(np.array(uav_xy) if uav_xy else np.empty((0, 2)))
    uav_scatter.set_color(uav_col)

    # Targets
    target_xy = []
    t_col = []

    for txt in target_texts:
        txt.remove()
    target_texts.clear()

    if not tg.empty:
        for _, row in tg.iterrows():
            tid = int(row["target_id"])
            x, y = float(row["target_x"]), float(row["target_y"])
            detected = bool(row.get("target_detected", False))

            if tid not in target_history:
                target_history[tid] = {"x": [], "y": []}

            # --- NEW: sliding trail ---
            target_history[tid]["x"].append(x)
            target_history[tid]["y"].append(y)
            target_history[tid]["x"] = target_history[tid]["x"][-TRAIL_LENGTH:]
            target_history[tid]["y"] = target_history[tid]["y"][-TRAIL_LENGTH:]

            target_xy.append([x, y])
            t_col.append(target_colors.get(tid, "red"))

            target_trails[tid].set_data(target_history[tid]["x"], target_history[tid]["y"])

            label = f"T{tid}"
            if detected:
                label += " (det)"
            txt = ax.text(x + 0.6, y + 0.6, label, fontsize=9, color="darkred")
            target_texts.append(txt)

        target_scatter.set_offsets(np.array(target_xy))
        target_scatter.set_color(t_col)
    else:
        target_scatter.set_offsets(np.empty((0, 2)))

    # Update detection stats
    detections = int(m["detected"].fillna(False).astype(bool).sum()) if "detected" in m.columns else 0

    recent_detections.append(detections)
    recent_detections[:] = recent_detections[-DETECTION_AVG_WINDOW:]   # keep last N

    avg_det = np.mean(recent_detections) if recent_detections else 0.0

    # Update info panel
    n_search = int((m["task_type"] == "search").sum()) if "task_type" in m.columns else 0
    n_track = int((m["task_type"] == "track").sum()) if "task_type" in m.columns else 0

    time_text.set_text(f"t = {int(t)}")
    info_text.set_text(
        f"search UAVs: {n_search}   tracking UAVs: {n_track}   "
        f"detections this frame: {detections}   avg detections: {avg_det:.3f}"
    )

    artists = [uav_scatter, target_scatter, time_text, info_text]
    artists.extend(list(uav_trails.values()))
    artists.extend(list(target_trails.values()))
    artists.extend(uav_texts)
    artists.extend(target_texts)
    return artists

# ------------------------------------------------------------
# Progress bar wrapper around FuncAnimation
# ------------------------------------------------------------
# print("Rendering frames...")
# for _ in tqdm(range(len(times)), desc="Building animation"):
#     pass

anim = FuncAnimation(
    fig,
    update,
    frames=len(times),
    interval=150,
    blit=False,
    repeat=False,
)

plt.close(fig)

# ------------------------------------------------------------
# Save animation
# ------------------------------------------------------------
saved = False
try:
    writer = FFMpegWriter(fps=8, bitrate=1800)
    anim.save(OUT_MP4, writer=writer)
    print(f"Saved MP4 to: {OUT_MP4}")
    saved = True
except Exception as e:
    print("FFmpeg save failed, falling back to GIF:", e)
    try:
        writer = PillowWriter(fps=8)
        anim.save(OUT_GIF, writer=writer)
        print(f"Saved GIF to: {OUT_GIF}")
        saved = True
    except Exception as e2:
        print("GIF save also failed:", e2)

Saved MP4 to: /content/swarm_artifacts/swarm_rollout.mp4


In [ ]:
from IPython.display import Video, Image, display

if saved and os.path.exists(OUT_MP4):
    display(Video(OUT_MP4, embed=True))
elif saved and os.path.exists(OUT_GIF):
    display(Image(filename=OUT_GIF))

In [ ]:
"""
TODO:
- Progress Bar
- Reduce trailing
- Add information about average detection rate
- Make t
- Add moving 3D obstacles
"""

'\nTODO:\n- Progress Bar\n- Reduce trailing\n- Add information about average detection rate\n- Make t\n- Add moving 3D obstacles\n'